# 分布式集合通信

你用 8 张卡训练，每张卡拿到不同的数据，算出了自己的梯度。下一步呢？

总不能各管各的，否则每张卡按自己的梯度更新，参数就分叉了，模型等于没训练。正确的做法是把 8 份梯度合成一份平均梯度，再发给每个人，让 8 张卡保持同步。

这一步在代码里是一行 `all_reduce`。背后是一套多卡之间传数据的固定动作，统称集合通信。这一节我们逐个看这些动作：谁发、谁收、怎么聚合、要传多少。读完你应该能自己回答：多卡之间的梯度同步，究竟靠哪个动作完成。


## 0. 先建立直觉：为什么需要集合通信

先想清楚一个问题：多卡训练到底难在哪？

难在一点：**每张卡各自计算，但结果必须全局一致**。以梯度同步为例，8 张卡各算出一份梯度，最终需要所有卡拿到相同的平均梯度，否则各自用优化器更新之后，权重就分叉了。

最朴素的做法是选出 0 号进程作为聚合点：其他 7 张卡把梯度都发给它，它求和后（得到平均梯度）再广播给所有人。语义上完全正确，但有一个问题：

**rank 0 成为瓶颈**。它要收 N-1 份、再发 N-1 份，通信量是其他卡的 2 倍；卡越多，rank 0 越慢，整个训练被它拖住。

所以集合通信要解决的核心问题就是：**怎么让所有卡又快又公平地完成「收集-合并-分发」**。

这一节你会看到，同样一件事，不同实现（朴素做法 vs 巧妙算法）通信量差很多。而这个「巧妙算法」，就是后面分布式训练框架真正用的东西。


### 先认识三个名词

看代码之前，先给三个词下定义，后面每个例子都会用到：

- **Rank**：每张卡（每个进程）的编号，从 0 开始。8 张卡就是 rank 0 ~ rank 7。
- **World size**：总共有多少张卡参与。
- **Process group（进程组）**：一次通信在哪些卡之间发生。默认是全部卡，也可以只指定一部分（比如 64 张卡里只让 8 张同步）。

为什么需要这三个词？因为它们构成了任意一次通信的「参与者清单」：rank 说明「我是谁」，world size 说明「总共有多少人」，process group 说明「这次谁参加」。搞不清这三个，后面所有动作都无从谈起。

这三个词很直白：rank 是进程编号，world size 是参与总数，process group 是「这次通信的参与集合」。


## 1. 最基础的动作：两个进程点对点传数据

集合通信的地基，是两个进程之间的**点对点通信**（send / recv）：一个进程把数据发给另一个进程。

PyTorch 里两个底层 API：

- `send`：把数据发给目标进程（对方收到之前，发送方一直等）；
- `recv`：从源进程收数据（数据到达之前，接收方一直等）。

这两个 API 平时训练代码里几乎不会直接出现——它们太底层，我们关心的是「多个进程共同完成一件事」的集合通信。但理解点对点，后面才好懂。

一个要知道的坑：阻塞的 send/recv 顺序不对会**死锁**。进程 A 想发给 B 再收 B 的数据，进程 B 也想先发给 A 再收 A 的数据，两边都在等对方先收，谁也动不了。实际框架用「非阻塞」版本避免这个问题，我们这里知道有这回事就行。


## 2. 七个标准动作：一个定义讲一个

从这一节开始，我们逐个看集合通信的标准动作。每个动作都用**一句话定义 + 输入/输出语义 + 一个具体用途 + 通信量**来讲解。

为什么是「一个定义讲一个」？因为这些动作彼此很像，区别只在三件事：**谁有数据、谁要结果、要不要聚合**。先给你一张地图，心里有个数。

先把七个动作的「一句话」列出来：

| 动作 | 一句话 | 输入 → 输出 | 典型场景 |
|:---|:---|:---|:---|
| broadcast | 一份数据复制给所有人 | 1 → all（相同） | 初始化时广播参数 |
| scatter | 大数组切片分给每人一份 | 1 → all（切片） | 分发数据 batch |
| gather | 各片收回到一个人拼起来 | all → 1 | 收集各卡结果 |
| reduce | 各卡数据聚合，结果只给一个人 | all → 1（聚合） | 聚合 loss |
| all-reduce | 各卡数据聚合，结果给每个人 | all → all（聚合） | DDP 同步梯度 |
| all-gather | 各片拼成完整版，人人都有 | all → all（拼接） | FSDP 拼参数 |
| reduce-scatter | 聚合后切成片，每人拿一片 | all → all（聚合切片） | ZeRO 切梯度 |

注意看「输入 → 输出」这一列：有的是 `1 → all`，有的是 `all → 1`，有的是 `all → all`。这一列就是它们最本质的区别。下面逐个来。


### 2.1 broadcast：一份数据复制给所有人

**定义**：rank 0 持有一份数据，把它复制给所有其他卡。语义是「1 份输入 → N 份相同输出」。

**输入/输出**：输入是 rank 0 上的一个数组；输出是每张卡上都有一份相同的副本。

**用途**：训练开始时，把初始模型参数从 rank 0 广播到所有卡，保证大家起点一致。

**通信量**：rank 0 要给其他 N-1 张卡各发一份，通信量 = (N-1) × 数据大小。

一句话记住它：**大家拿到的是完全相同的副本**。


In [ ]:
# === broadcast 的 numpy 模拟 ===
import numpy as np

# rank 0 持有原始数据
np.random.seed(0)
data_on_rank0 = np.array([10, 20, 30, 40])

# broadcast 前：只有 rank 0 有数据，其他卡是空的
cards_before = [None, None, None, None]
cards_before[0] = data_on_rank0.copy()
print("broadcast 前：")
for rank, data in enumerate(cards_before):
    print(f"  rank {rank}: {data}")

# broadcast 后：每张卡都有一份相同的副本
cards_after = [data_on_rank0.copy() for _ in range(4)]
print("\nbroadcast 后：")
for rank, data in enumerate(cards_after):
    print(f"  rank {rank}: {data}")

print("\n关键观察：1 份输入 → N 份相同输出。")
print("通信量 = 数据大小 × (N-1)（rank 0 要发给其他 N-1 张卡）。")

### 2.2 scatter：大数组切成 N 片分发

**定义**：rank 0 持有 N 份数据，把第 i 份发给第 i 张卡。语义是「1 份大数组 → N 份不同切片」。

**输入/输出**：输入是 rank 0 上的一个 N 片大数组；输出是每张卡各拿其中一片。

**和 broadcast 的区别**：broadcast 大家拿到的**完全相同**；scatter 大家拿到的**各不相同**（每人一片）。同样是 1 → all，但一个发的是同一个副本，一个发的是不同切片。

**用途**：rank 0 读了一个大 batch 的数据，切成 N 份分给 N 张卡做数据并行。

**通信量**：rank 0 发 N-1 份，每份大小 = 数据大小/N，通信量 = (N-1) × D/N。

一句话记住它：**大家拿到的各不相同，每人一片**。


In [ ]:
# === scatter 的 numpy 模拟 ===
import numpy as np

# rank 0 持有一份大数组（4 行，准备切成 4 片）
big_array = np.array([[1, 2],
                      [3, 4],
                      [5, 6],
                      [7, 8]])

print("scatter 前：")
print(f"  rank 0 (完整):\n{big_array}")
for rank in range(1, 4):
    print(f"  rank {rank}: 空")

# scatter：沿 axis=0 切成 4 片，第 i 片发给第 i 张卡
shards = np.split(big_array, 4, axis=0)
print("\nscatter 后：")
for rank, shard in enumerate(shards):
    print(f"  rank {rank}: {shard.ravel()}")

print("\n关键观察：1 份大数组 → N 份不同切片，每卡拿到 1/N。")
print("对比 broadcast：那里大家拿一样的；这里大家拿不一样的一人一片。")

### 2.3 gather：各片收回到一个人拼起来

**定义**：每张卡持有一片数据，rank 0 把所有片按顺序拼成完整数组。语义是「N 份输入 → 1 份拼接输出」——是 scatter 的反向操作。

**输入/输出**：输入是每张卡上的一片；输出是 rank 0 上的完整拼接数组。

**注意**：gather 之后，**只有 rank 0 有完整结果**，其他卡还拿着自己的原数据。

**用途**：分布式推理时，把每张卡生成的 token 序列收回到 rank 0 做后处理。

**通信量**：rank 0 从其他 N-1 张卡各收一片，通信量 = (N-1) × D/N。

一句话记住它：**数据往一个方向汇聚，结果只落在 rank 0**。


In [ ]:
# === gather 的 numpy 模拟 ===
import numpy as np

# 每张卡持有一片数据
shards = [np.array([10, 20]),
          np.array([30, 40]),
          np.array([50, 60]),
          np.array([70, 80])]

print("gather 前：")
for rank, s in enumerate(shards):
    print(f"  rank {rank}: {s}")

# gather：rank 0 收集所有片，按顺序拼接
gathered_on_rank0 = np.concatenate(shards, axis=0)
print("\ngather 后：")
print(f"  rank 0 (拼接): {gathered_on_rank0}")
for rank in range(1, 4):
    print(f"  rank {rank}: 仍持有原数据 {shards[rank]}（未拼接）")

print("\n关键观察：N 份输入 → 1 份拼接输出，而且只在 rank 0 上。")

### 2.4 reduce：各卡数据聚合，结果只给一个人

**定义**：每张卡持有一份数据，按某种运算（通常是求和）聚合，结果只保留在 rank 0。语义是「N 份输入 → 1 份聚合输出」——比 gather 多一步「按位聚合」。

**输入/输出**：输入是每张卡上的一个数组；输出是 rank 0 上按位聚合后的数组。

**用途**：计算 loss 时，每张卡有自己的 loss，reduce 到 rank 0 看总 loss。注意：如果**每张卡都需要**最终结果，就要用下一个动作 all-reduce 了。

**通信量**：与 gather 相同，rank 0 收 N-1 份，通信量 = (N-1) × D/N。

一句话记住它：**和 gather 一样只落在 rank 0，但多一步「按位聚合」**。


In [ ]:
# === reduce 的 numpy 模拟 ===
import numpy as np

# 每张卡持有一个 tensor
cards = [np.array([1.0, 2.0]),
         np.array([3.0, 4.0]),
         np.array([5.0, 6.0]),
         np.array([7.0, 8.0])]

print("reduce 前：")
for rank, data in enumerate(cards):
    print(f"  rank {rank}: {data}")

# reduce sum：对应位置相加，结果只在 rank 0
result_on_rank0 = np.sum(np.stack(cards), axis=0)
print("\nreduce sum 后：")
print(f"  rank 0: {result_on_rank0}")
for rank in range(1, 4):
    print(f"  rank {rank}: 没保留结果")

print("\n关键观察：N 份输入 → 1 份聚合输出，只在 rank 0。")
print("求和是最常用的聚合运算（还能用 max / min 等）。")

### 2.5 all-reduce：各卡数据聚合，结果给每个人

**定义**：每张卡持有一份数据，聚合（求和）后，**每张卡都拿到完整结果**。语义是「N 份不同输入 → N 份相同聚合输出」。等价于「先 reduce 到 rank 0，再 broadcast 给所有人」。

**输入/输出**：输入是每张卡上的一个数组；输出是每张卡上都有一份相同的聚合数组。

**用途**：这是分布式训练里出现频率最高的动作。DDP 反向传播结束时调用一次 all-reduce，让每张卡拿到相同的平均梯度，这样每张卡的优化器更新后，权重依然一致。

下面先用「朴素实现」（集中到 rank 0 再广播）演示语义。至于它为什么在工程上要做得更巧妙，下一节专门讲。

一句话记住它：**和 reduce 一样聚合，但结果人人都有**。


In [ ]:
# === all-reduce 的 numpy 模拟（先看语义，算法下一节） ===
import numpy as np

cards = [np.array([1.0, 2.0]),
         np.array([3.0, 4.0]),
         np.array([5.0, 6.0]),
         np.array([7.0, 8.0])]

print("all-reduce 前：")
for rank, data in enumerate(cards):
    print(f"  rank {rank}: {data}")

# all-reduce sum：每张卡都得到完整的求和结果
total = np.sum(np.stack(cards), axis=0)
cards_after = [total.copy() for _ in range(4)]

print("\nall-reduce sum 后：")
for rank, data in enumerate(cards_after):
    print(f"  rank {rank}: {data}")

print("\n关键观察：N 份不同输入 → N 份相同聚合输出。")
print("和 reduce 的区别：结果人人都有，不只是 rank 0。")

### 2.6 all-gather：各片拼成完整版，人人都有

**定义**：每张卡持有一片数据，操作后**每张卡都拿到所有片的完整拼接**。语义是「N 份输入 → N 份相同拼接输出」。

**输入/输出**：输入是每张卡上的一片；输出是每张卡上都有一份完整拼接数组。

**和 gather 的区别**：gather 只有 rank 0 有完整结果；all-gather 人人都有。

**用途**：FSDP 前向传播前，用 all-gather 把分片存着的参数拼成完整的一层，用完再丢。

**通信量**：每张卡要收到除了自己以外的 N-1 片，通信量 = (N-1)/N × D。

一句话记住它：**和 gather 一样拼接，但结果人人都有**。


In [ ]:
# === all-gather 的 numpy 模拟 ===
import numpy as np

# 每张卡持有一片
shards = [np.array([10, 20]),
          np.array([30, 40]),
          np.array([50, 60]),
          np.array([70, 80])]

print("all-gather 前：")
for rank, s in enumerate(shards):
    print(f"  rank {rank}: {s}")

# all-gather：每张卡都得到所有片的拼接
full = np.concatenate(shards, axis=0)
cards_after = [full.copy() for _ in range(4)]

print("\nall-gather 后：")
for rank, data in enumerate(cards_after):
    print(f"  rank {rank}: {data}")

print("\n关键观察：和 gather 的区别——每张卡都有完整结果，不只 rank 0。")

### 2.7 reduce-scatter：各卡聚合后，每人拿一片

**定义**：每张卡持有一份完整数据，聚合（求和）后**切成 N 片，每人只拿一片**。语义是「N 份完整输入 → N 份聚合切片输出」。

**输入/输出**：输入是每张卡上的完整数组；输出是每张卡上各有一片聚合后的切片。

**一个关键关系**：`reduce-scatter` + `all-gather` = `all-reduce`。先各算各的再拼起来，正好等于人人拿到完整结果。这个关系后面讲 ring 算法时直接用。

**用途**：ZeRO 训练里，每张卡只保留自己负责的那片参数的梯度，其余梯度立即释放省显存。

**通信量**：每张卡发 N-1 片、每片 1/N，通信量 = (N-1)/N × D。

一句话记住它：**先聚合，再切成片，每人只留一片**。


In [ ]:
# === reduce-scatter 的 numpy 模拟 ===
import numpy as np

N = 4
# 每张卡持有一份完整 vector（长度 = N）
cards = [np.array([1.0, 2.0, 3.0, 4.0]),
         np.array([10., 20., 30., 40.]),
         np.array([100., 200., 300., 400.]),
         np.array([0.1, 0.2, 0.3, 0.4])]

print("reduce-scatter 前：")
for rank, data in enumerate(cards):
    print(f"  rank {rank}: {data}")

# 先对所有卡求和，再切成 N 片
stacked = np.stack(cards)          # 形状 (N, N)
summed = stacked.sum(axis=0)       # 每个位置都是所有卡的求和
shards = np.split(summed, N)       # 切成 N 份

print("\n所有卡求和后：", summed)
print("\nreduce-scatter 后：")
for rank, shard in enumerate(shards):
    print(f"  rank {rank}: {shard}")

print("\n关键观察：N 份完整输入 → N 份聚合切片输出，每卡只有 1/N。")

### 2.8 七个动作总览

把七个动作放在一张表里对照着看，方向、通信量、典型场景一目了然。先看表，再想想它们之间有没有规律。


In [ ]:
# === 七个集合通信对比表 ===
rows = [
    ("算子",          "方向",                    "通信量",          "典型场景"),
    ("broadcast",     "1 → all (相同)",          "(N-1) × D",       "初始化时广播参数"),
    ("scatter",       "1 → all (切片)",          "(N-1) × D/N",     "rank 0 分发 batch"),
    ("gather",        "all → 1",                 "(N-1) × D/N",     "rank 0 收集结果"),
    ("reduce",        "all → 1 (聚合)",          "(N-1) × D/N",     "rank 0 聚合 loss"),
    ("all-reduce",    "all → all (聚合)",        "2(N-1)/N × D",    "DDP 同步梯度"),
    ("all-gather",    "all → all (拼接)",        "(N-1)/N × D",     "FSDP 拼参数"),
    ("reduce-scatter", "all → all (聚合切片)",   "(N-1)/N × D",     "ZeRO 切梯度"),
]

widths = [16, 24, 16, 24]
for row in rows:
    line = " | ".join(f"{c:<{w}}" for c, w in zip(row, widths))
    print(line)
    if row[0] == "算子":
        print("-" * len(line))

print("\nD = 单卡数据大小，N = 卡数。")
print("通信量里那些 (N-1)/N、2(N-1)/N 的系数，来自下一节的 ring 算法。")
print("现在只需要记住：all-to-all 是后面 MoE 的核心，这里先留个位。")

## 3. all-reduce 的巧妙算法：ring（环形拓扑）

还记得第 0 节的问题吗：rank 0 作为聚合点，要收 N 份、发 N 份，成为瓶颈。

有没有更好的办法？有。它叫 **ring all-reduce**（环形拓扑）。

思路一句话：**所有卡排成一个环，数据沿环每步只传给相邻的卡。**

- 阶段一（reduce-scatter）：每张卡把自己的数据切成 N 片，每轮把一片传给右边的卡，同时把左边卡传来的值累加到自己对应位置；重复 N-1 轮后，每张卡手里恰好有一片是全局聚合和。
- 阶段二（all-gather）：把这片「全局和」沿环再传 N-1 轮，让每张卡都收到所有片，最终每张卡都持有完整聚合结果。

好处是什么？**没有瓶颈**。集中式方案里 rank 0 要收 N 份、发 N 份；ring 方案里每张卡都只和左右邻居通信一次，负载均匀。

下面用 N=4 把整个过程一步步算给你看。先看它为什么对，再上代码。


### 3.1 手算：4 张卡围成一环

假设 4 张卡每人手里有一个 4 元素向量，要算**对应位置的 4 项之和**，每张卡得到结果。

每张卡把向量看成 4 片（每片 1 个元素），然后按下面的约定传递：

**阶段一（reduce-scatter）**：共 3 轮，每轮把某一片传给右边的卡累加。3 轮之后，每张卡手上恰好有一片是全局聚合和。

**阶段二（all-gather）**：共 3 轮，把这片「全局和」沿环传给所有卡。3 轮之后，每张卡手里 4 片全是全局聚合和。

直接看代码跑一遍，每一步都打印出来。可以先用纸面想一个数字验证：第 1 轮时，rank 0 到底把第几片传给 rank 1？代码会告诉你答案。


In [ ]:
# === ring all-reduce 逐步演示（N=4） ===
import numpy as np

N = 4
np.random.seed(42)
# 每张卡持有长度 4 的向量（切成 4 片，每片 1 个元素）
cards = [list(np.random.randint(1, 10, size=N)) for _ in range(N)]

print("初始：每张卡持有自己的向量")
for r in range(N):
    print(f"  rank {r}: {cards[r]}")

expected = [sum(cards[r][k] for r in range(N)) for k in range(N)]
print(f"\n期望结果（每卡对应位置之和）：{expected}")

# ======== 阶段一：reduce-scatter（3 轮，每轮传 1 片给右边并累加） ========
buffer = [row[:] for row in cards]
print("\n--- 阶段一 reduce-scatter：每轮把一片传给右边的卡累加 ---")
for step in range(N - 1):
    # 记录本轮每张卡要传出去的片
    outgoing = {}
    for r in range(N):
        send_idx = (r - step) % N
        outgoing[r] = (send_idx, buffer[r][send_idx])
    # 统一执行：右邻居把收到的值加到自己对应位置
    for r in range(N):
        send_idx, value = outgoing[r]
        right = (r + 1) % N
        buffer[right][send_idx] += value
    print(f"  第 {step+1} 轮后：")
    for r in range(N):
        print(f"    rank {r}: {buffer[r]}")

# 验证：此时 rank r 手上第 (r+1)%N 片是全局和
print("\n阶段一结果：每张卡持有一片全局聚合和")
for r in range(N):
    master_idx = (r + 1) % N
    got = buffer[r][master_idx]
    assert got == expected[master_idx], f"rank {r} 片 {master_idx} = {got}"
    print(f"  rank {r} 持有片 {master_idx} = {got} ✓")

# ======== 阶段二：all-gather（3 轮，把「和」片传给所有人） ========
# 每张卡当前要传出去的片：自己手上那片全局和
out_chunk_idx = [(r + 1) % N for r in range(N)]
out_value = [buffer[r][(r + 1) % N] for r in range(N)]

print("\n--- 阶段二 all-gather：把每张卡的「和」片沿环传一圈 ---")
for step in range(N - 1):
    snapshot_idx = out_chunk_idx[:]
    snapshot_val = out_value[:]
    for r in range(N):
        right = (r + 1) % N
        received_idx = snapshot_idx[r]
        received_val = snapshot_val[r]
        buffer[right][received_idx] = received_val
        # 右邻居下一轮要把刚收到的片继续传出去
        out_chunk_idx[right] = received_idx
        out_value[right] = received_val
    print(f"  第 {step+1} 轮后：")
    for r in range(N):
        print(f"    rank {r}: {buffer[r]}")

# 最终验证
all_ok = all(buffer[r] == expected for r in range(N))
print(f"\n最终结果（每卡是否都是全局和？）：{all_ok}")
for r in range(N):
    print(f"  rank {r}: {buffer[r]}")

print("\n关键观察：")
print(f"  两个阶段各 {N-1} 轮，每轮每卡只传 1 个元素。")
print(f"  每卡总通信量 = 2 × (N-1) 个元素，和卡数 N 关系不大。")
print(f"  而集中式方案里 rank 0 要收 4 个、发 4 个 = 8 个，成为瓶颈。")

### 3.2 ring 为什么快：没有瓶颈

把两种方案再对比一下：

| 方案 | 通信总量（每卡） | 负载分布 |
|:---|:---|:---|
| 集中式（rank 0 聚合） | rank 0 传 2N 份，其他卡传 1 份 | rank 0 是瓶颈，N 越大越堵 |
| ring 方案 | 每卡都传 2(N-1)/N 份 | 负载均匀，没有瓶颈 |

具体数字：数据总大小是 D，N 张卡。

- 集中式：rank 0 收 (N-1)×D、再发 (N-1)×D，通信量 2(N-1)×D；
- ring 方案：每卡只传 2(N-1)/N × D，N 越大，人均负担越接近「2D」这个下限。

这就是为什么分布式训练框架都内置 ring 算法：**卡越多，集中式越慢，ring 几乎不涨。**

顺带一提，ring 的核心思想「把一个大任务切成小片，沿环轮流传递」，在后面 all-gather、reduce-scatter 里也是同一个套路。


## 4. 还没讲完的动作：all-to-all（数据重排）

上面七个动作，还有一个重要的漏网之鱼：**all-to-all**。

**定义**：每张卡手里有 N 份数据，第 i 份本来就打算给第 i 张卡。操作之后，第 i 张卡收到「所有卡发给它的那一份」。语义是「数据在卡与卡之间做一次整体重排」。

**为什么单独放一节**：它的通信模式和前面都不一样，而且无法用 ring 优化。

**它出现在哪里**：MoE 模型（混合专家）。MoE 里每个 token 会被路由到某个「专家」计算，而专家分散在不同卡上——token 就得从自己的卡跑到专家所在的卡，这就是 all-to-all。

这里先认识它，5D 并行那节会用它把 MoE 讲透。


In [ ]:
# === all-to-all 的 numpy 模拟 ===
import numpy as np

N = 4
# 每张卡的发送 buffer：第 i 份是给 rank i 的
# 每个元素 (sender, receiver)，方便看清楚谁发给谁
send_buffers = [
    np.array([[0, 0], [0, 1], [0, 2], [0, 3]]),  # rank 0 的 4 份：给 0,1,2,3
    np.array([[1, 0], [1, 1], [1, 2], [1, 3]]),  # rank 1 的 4 份
    np.array([[2, 0], [2, 1], [2, 2], [2, 3]]),  # rank 2 的 4 份
    np.array([[3, 0], [3, 1], [3, 2], [3, 3]]),  # rank 3 的 4 份
]

print("all-to-all 前（每卡的发送数组，第 i 行目标 rank i）：")
for rank, buf in enumerate(send_buffers):
    print(f"  rank {rank}:")
    print(buf)

# all-to-all 等价于把 (sender, receiver) 二维表转置
all_data = np.transpose(np.stack(send_buffers), (1, 0, 2))

print("\nall-to-all 后（每卡收到的数组，第 j 行来自 rank j）：")
for rank in range(N):
    print(f"  rank {rank}:")
    print(all_data[rank])

print("\n关键观察：所有卡同时在互相换数据，每张卡都发 N 份、收 N 份。")
print("通信量 = N × 数据大小——无法用 ring 分摊，这是它贵的原因。")

## 5. 进阶：怎么用一句话描述「数据怎么切」

读并行论文（Megatron、DeepSeek 的技术报告）时，常看到一种紧凑的写法描述「张量在多卡上怎么分布」。这一节用最简单的方式把它讲清楚。

假设你要把一个矩阵 A 分给 4 张卡。可以按行切（每卡拿几行），也可以按列切（每卡拿几列）。怎么用记号表达「我切的是哪个方向」？

约定：**矩阵的轴叫 I（行）、J（列）；卡组成一个网格，轴叫 X、Y。**

- 写 `A[I_X, J]`：表示 A 的**行**沿卡的 **X** 方向切开（每卡拿 I/X 行），列不切、人人有完整列。
- 写 `A[I, J_Y]`：表示 A 的**列**沿卡的 **Y** 方向切开，行不切。

**记住一条规则：下标出现的轴 = 切分；没出现的轴 = 复制。**

下面用一个 2×2 的卡网格（4 张卡）实际演示两种切法，感受记号到底在说什么。


In [ ]:
# === partition notation：A[I_X, J] vs A[I, J_Y] ===
import numpy as np

# 完整矩阵 A（4×4），要放到 2×2 = 4 张卡上
A = np.arange(16).reshape(4, 4)
print("完整矩阵 A (shape (4,4)):")
print(A)
print()

print("=== A[I_X, J]：行沿 X 切，列复制 ===")
print("X 方向有 2 个位置，所以行切成 2 份（每份 2 行）")
row_shards = np.split(A, 2, axis=0)
for x in range(2):
    for y in range(2):
        print(f"  device (x={x}, y={y}):")
        print(row_shards[x])

print("=== A[I, J_Y]：列沿 Y 切，行复制 ===")
col_shards = np.split(A, 2, axis=1)
for x in range(2):
    for y in range(2):
        print(f"  device (x={x}, y={y}):")
        print(col_shards[y])

print("\n关键观察：下标出现 = 切分，不出现 = 复制。")
print("同一份数据，切法不同，每张卡看到的子矩阵完全不同。")

### 5.1 用这个记号描述集合通信

有了记号，几个动作可以一句话写清楚：

- **all-gather**：`A[I, J_Y] → A[I, J]`，去掉「列切分」，恢复成完整矩阵。
- **reduce-scatter**：`A[I, J] → A[I, J_Y]`，把完整矩阵聚合后切成列。
- **all-reduce**：等价于「先 reduce-scatter 再 all-gather」，即 `A[I, J_Y] → A[I, J_Y]`（内容变了）。

**对照 2.6、2.7**：all-gather 是把「切分」去掉（分散数据拼回完整），reduce-scatter 是给「完整」加上切分（聚合后切成片）。记这个记号，比记一堆文字省力。


## 6. 实战：torch.distributed 里怎么调用

前面全程用 numpy 模拟，接下来用 PyTorch 的真实 API 跑一次。动作语义完全不变，只是把调用方式交给 PyTorch。PyTorch 的 `torch.distributed`（简称 `dist`）封装了底层集合通信库，GPU 上用 NCCL 后端，CPU 上可以用 gloo 后端。

核心流程三步：

1. **初始化**：每个进程调用 `dist.init_process_group(...)`，声明自己的 rank 和 world size；
2. **调用集合操作**：如 `dist.all_reduce(tensor)`，注意它是**原地修改** tensor；
3. **收尾**：`dist.destroy_process_group()`。

下面这段代码会真的启动 2 个进程，用 gloo 后端跑一次 all-reduce。（在 Jupyter 里跑 multiprocessing 可能不太友好，读懂逻辑即可，不必真的逐行跑。）


In [ ]:
# === torch.distributed 真实 2 进程 all-reduce 示例 ===
# 单机启动 2 个进程，gloo 后端（CPU 通信，无需 GPU）
# 在 Jupyter 里跑 multiprocessing 可能不友好，读懂逻辑即可

import os
import torch
import torch.multiprocessing as mp
import torch.distributed as dist


def worker_fn(rank, world_size):
    """每个子进程执行这个函数：初始化 → all-reduce → 清理。"""
    # 1. 初始化进程组（env:// 表示从环境变量读地址）
    os.environ["MASTER_ADDR"] = "127.0.0.1"
    os.environ["MASTER_PORT"] = "29501"
    dist.init_process_group(backend="gloo", rank=rank, world_size=world_size)

    # 2. 每张卡持有一个 tensor（rank 0 持有 [1,2]，rank 1 持有 [2,3]）
    tensor = torch.tensor([float(rank + 1), float(rank + 2)])
    print(f"[before] rank {rank}: {tensor.tolist()}")

    # 3. all-reduce 求和（原地操作！tensor 被直接改写）
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    print(f"[after]  rank {rank}: {tensor.tolist()}  （期望 [3.0, 5.0]）")

    # 4. 清理
    dist.destroy_process_group()


# 启动 2 个进程
if __name__ == "__main__":
    world_size = 2
    mp.start_processes(worker_fn, args=(world_size,), nprocs=world_size,
                       join=True, start_method="fork")
    print("\n两个 rank 的 all-reduce 结果都是 [3.0, 5.0]。")

In [ ]:
# === torch.distributed 常用 API 速查 ===
api_reference = """
# 初始化（每个进程启动时调用一次）
dist.init_process_group(backend="nccl" | "gloo", rank=0, world_size=8)

# all-reduce：原地修改 tensor，所有人拿到聚合结果
dist.all_reduce(tensor, op=dist.ReduceOp.SUM)

# broadcast：从 src 广播给所有人
dist.broadcast(tensor, src=0)

# all-gather：输出是列表，长度 = world_size
out_list = [torch.empty_like(tensor) for _ in range(world_size)]
dist.all_gather(out_list, tensor)

# reduce-scatter：输入是列表，输出是单个 tensor
in_list = [torch.rand(4) for _ in range(world_size)]
out = torch.empty(4)
dist.reduce_scatter(out, in_list, op=dist.ReduceOp.SUM)

# all-to-all（single tensor 版，MoE 路由常用）
out = torch.empty_like(input_tensor)
dist.all_to_all_single(out, input_tensor)
"""
print(api_reference)
print("记忆要点：")
print("  - all_reduce / broadcast 是原地操作，tensor 直接被改")
print("  - all_gather / reduce_scatter 用列表中转")
print("  - 所有操作默认用 SUM（求和）")

## 小结

这一节的逻辑链：

1. 多卡要配合，就得互相传数据 → 有了集合通信这组标准动作；
2. 七个动作的本质区别是「谁有数据、谁要结果、要不要求和」；
3. 同样的语义，实现算法天差地别——ring 让 all-reduce 摆脱集中式的瓶颈；
4. all-to-all 是特殊的一种，无法用 ring 优化，留给 MoE 那节。

确认你已经搞懂：

- [ ] rank / world size / process group 是三个基本概念
- [ ] broadcast：1 → all 相同副本；scatter：1 → all 不同切片
- [ ] gather：all → 1 拼接（只在 rank 0）；reduce：all → 1 聚合（只在 rank 0）
- [ ] all-reduce：all → all 聚合，人人有结果（DDP 靠它同步梯度）
- [ ] all-gather：all → all 拼接（FSDP 靠它拼参数）
- [ ] reduce-scatter：all → all 聚合切片（ZeRO 靠它省显存）
- [ ] reduce-scatter + all-gather = all-reduce
- [ ] ring 算法让 all-reduce 每卡通信量约等于 2×数据量，与卡数无关
- [ ] all-to-all 是整体重排，无法用 ring 优化，是 MoE 的核心通信
- [ ] partition notation：下标出现的轴 = 切分，没出现 = 复制

## 作业

> 可以用 AI 询问思路、拆步骤、检查方向，但不建议直接让 AI「做完这道题」。

**作业 1：手算 ring all-reduce 的通信量**

一个 64 MB 的 tensor 在 8 张卡上做 ring all-reduce（求和）。
每张卡发送+接收的总数据量是多少 MB？如果是集中式方案（rank 0 收齐再广播），
rank 0 一个人的通信量又是多少 MB？

小提示：ring 分两个阶段，每阶段 N-1 轮，每轮每卡传 D/N。
集中式方案：rank 0 收 (N-1)×D 再发 (N-1)×D。

In [ ]:
# 作业 1：ring all-reduce vs 集中式方案
D_mb = 64    # tensor 大小 64 MB
N = 8        # 8 张卡

# TODO: 每卡通信量（MB）
ring_per_card = None

# TODO: 集中式方案里 rank 0 的通信量（MB）
naive_rank0 = None

assert ring_per_card is not None and naive_rank0 is not None, "请先计算两个值"
# ring = reduce-scatter + all-gather，每阶段 (N-1) 轮 × D/N
expected_ring = 2 * (N - 1) * D_mb / N
# 集中式：rank 0 收 (N-1)*D 再发 (N-1)*D
assert abs(ring_per_card - expected_ring) < 0.1, f"ring 应为 {expected_ring:.1f} MB"
assert abs(naive_rank0 - 2 * (N - 1) * D_mb) < 0.1, \
    f"集中式方案 rank 0 应为 {2 * (N - 1) * D_mb} MB"

print(f"作业 1 通过！")
print(f"  ring：每卡 {ring_per_card:.1f} MB，负载均匀")
print(f"  集中式：rank 0 一人 {naive_rank0} MB，是瓶颈")
print(f"  卡越多，集中式的负担线性增长，ring 几乎不涨。")

**作业 2：用记号描述 FSDP 前向**

FSDP（ZeRO-3 的一种实现）前向时，参数 W 分片存在 N 张卡上（沿 X 方向切，记为 `W[I, J_X]`），
使用前先 all-gather 拼成完整参数。请填空：

小提示：对照 5.1 节的例子——all-gather 的作用是「去掉切分」。

In [ ]:
# 作业 2：FSDP 前向 all-gather 的 partition notation

# TODO: 前向之前，参数 W 的切分形式（沿 X 切）
fsdp_input = None

# TODO: all-gather 之后，参数 W 的形式（不再切分）
fsdp_output = None

assert fsdp_input == "W[I, J_X]", "FSDP 前向之前参数沿 X 分片，应为 W[I, J_X]"
assert fsdp_output == "W[I, J]", "all-gather 去掉切分，应为 W[I, J]"

print("作业 2 通过！")
print(f"  前向之前：{fsdp_input}（每卡只存 1/N 的参数）")
print(f"  all-gather 后：{fsdp_output}（每卡临时持有完整参数）")
print(f"  用完反向时再 reduce-scatter 切回去——和 5.1 的记号一致。")

**作业 3：MoE 的 all-to-all 通信量**

一个 MoE 模型，batch=512，seq=2048，hidden=4096，每个 token 激活 top_k=2 个 expert，BF16。
专家并行度 EP=4 和 EP=8 时，每张卡**一个 MoE 层**要收发多少 GB？

小提示：每卡每次 all-to-all 发送 = (batch × seq ÷ EP) × top_k × hidden × 2 字节；
每个 MoE 层有 2 次 all-to-all（token 发出去 + 结果发回来）。

In [ ]:
# 作业 3：不同 EP 并行度下的 all-to-all 通信量
batch = 512
seq = 2048
hidden = 4096
top_k = 2
bytes_elem = 2   # BF16


def ep_all_to_all_gb(n_ep):
    """一个 MoE 层里，每张卡的 all-to-all 总通信量（GB）。"""
    # TODO: 在这里实现
    return None


comm_4 = ep_all_to_all_gb(4)
comm_8 = ep_all_to_all_gb(8)

assert comm_4 is not None and comm_8 is not None, "请先实现 ep_all_to_all_gb"


def expected(n_ep):
    tokens_per_card = batch * seq / n_ep
    bytes_per_send = tokens_per_card * top_k * hidden * bytes_elem
    return 2 * bytes_per_send / 1e9   # 一层 2 次 all-to-all


assert abs(comm_4 - expected(4)) < 0.01, f"EP=4 应为 {expected(4):.2f} GB"
assert abs(comm_8 - expected(8)) < 0.01, f"EP=8 应为 {expected(8):.2f} GB"

print(f"作业 3 通过！")
print(f"  EP=4：一层单卡 all-to-all {comm_4:.2f} GB")
print(f"  EP=8：一层单卡 all-to-all {comm_8:.2f} GB")
print(f"  EP 翻倍，每卡通信量减半（{comm_8/comm_4*100:.0f}%）——")
print(f"  但 all-to-all 的点对点数量随卡数变多，所以 EP 不是越大越好。")

## 参考资料

- [NCCL Documentation](https://docs.nvidia.com/deeplearning/nccl/) — NVIDIA 的 GPU 集合通信库
- [PyTorch torch.distributed](https://pytorch.org/docs/stable/distributed.html) — 官方 API 文档
- Sergeev & Del Balso, [Horovod: fast and easy distributed deep learning in TensorFlow](https://arxiv.org/abs/1802.05799), 2018 — ring all-reduce 的工程实现参考
- Rajbhandari et al., [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054), 2020 — 用到了 reduce-scatter 与 all-gather
- Shoeybi et al., [Megatron-LM](https://arxiv.org/abs/1909.08053), 2019 — tensor parallelism 与 partition notation
- Jiang et al., [DeepSeek-MoE](https://arxiv.org/abs/2401.06066), 2024 — MoE 的 all-to-all 通信